# Challenge 1 - Tic Tac Toe

In this lab you will perform deep learning analysis on a dataset of playing [Tic Tac Toe](https://en.wikipedia.org/wiki/Tic-tac-toe).

There are 9 grids in Tic Tac Toe that are coded as the following picture shows:

![Tic Tac Toe Grids](tttboard.jpg)

In the first 9 columns of the dataset you can find which marks (`x` or `o`) exist in the grids. If there is no mark in a certain grid, it is labeled as `b`. The last column is `class` which tells you whether Player X (who always moves first in Tic Tac Toe) wins in this configuration. Note that when `class` has the value `False`, it means either Player O wins the game or it ends up as a draw.

Follow the steps suggested below to conduct a neural network analysis using Tensorflow and Keras. You will build a deep learning model to predict whether Player X wins the game or not.

## Step 1: Data Engineering

This dataset is almost in the ready-to-use state so you do not need to worry about missing values and so on. Still, some simple data engineering is needed.

1. Read `tic-tac-toe.csv` into a dataframe.
1. Inspect the dataset. Determine if the dataset is reliable by eyeballing the data.
1. Convert the categorical values to numeric in all columns.
1. Separate the inputs and output.
1. Normalize the input data.

In [1]:
from pathlib import Path
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from IPython.display import display

SEED = 42
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()
DATA_DIR = Path.cwd() if Path("tic-tac-toe.csv").exists() else Path("your-code")
df = pd.read_csv(DATA_DIR / "tic-tac-toe.csv")
display(df.head(), df.describe(include="all"))
print("Shape:", df.shape, "Missing:", df.isna().sum().sum(), "Duplicates:", df.duplicated().sum())
display(df["class"].value_counts())
board_columns = list(df.columns[:-1])
assert not df.isna().any().any()
assert not df.duplicated().any()
assert set(df[board_columns].to_numpy().ravel()) == {"x", "o", "b"}
y = df["class"].astype(str).str.lower().map({"true": 1, "false": 0}).to_numpy(dtype="int32")
# Audit labels against the eight winning lines; these are NOT model inputs.
lines = [(0,1,2),(3,4,5),(6,7,8),(0,3,6),(1,4,7),(2,5,8),(0,4,8),(2,4,6)]
boards = df[board_columns].to_numpy()
x_wins = np.array([any(all(row[j] == "x" for j in line) for line in lines) for row in boards])
assert np.array_equal(x_wins.astype(int), y)
counts_x = (boards == "x").sum(axis=1)
counts_o = (boards == "o").sum(axis=1)
assert np.isin(counts_x - counts_o, [0, 1]).all()
# One-hot encoding avoids inventing an ordering between x, o and blank.
# Fixed domain categories need no fitting and every input is already in [0, 1].
X = np.stack([(boards == mark).astype("float32") for mark in ("b", "o", "x")], axis=-1).reshape(-1, 27)
print("Encoded shape:", X.shape, "Input range:", (X.min(), X.max()))
print("All labels match X's winning lines; board counts are consistent with alternating turns.")

/Users/mithila/AI-Engineering/week6/lab/lab-neural-networks/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


,TL,TM,TR,ML,MM,MR,BL,BM,BR,class
0,x,x,x,x,o,o,x,o,o,True
1,x,x,x,x,o,o,o,x,o,True
2,x,x,x,x,o,o,o,o,x,True
3,x,x,x,x,o,o,o,b,b,True
4,x,x,x,x,o,o,b,o,b,True


,TL,TM,TR,ML,MM,MR,BL,BM,BR,class
count,958,958,958,958,958,958,958,958,958,958
unique,3,3,3,3,3,3,3,3,3,2
top,x,x,x,x,x,x,x,x,x,True
freq,418,378,418,378,458,378,418,378,418,626


Shape: (958, 10) Missing: 0 Duplicates: 0


class
True     626
False    332
Name: count, dtype: int64

Encoded shape: (958, 27) Input range: (np.float32(0.0), np.float32(1.0))
All labels match X's winning lines; board counts are consistent with alternating turns.


## Step 2: Build Neural Network

To build the neural network, you can refer to your own codes you wrote while following the [Deep Learning with Python, TensorFlow, and Keras tutorial](https://www.youtube.com/watch?v=wQ8BIBpya2k) in the lesson. It's pretty similar to what you will be doing in this lab.

1. Split the training and test data.
1. Create a `Sequential` model.
1. Add several layers to your model. Make sure you use ReLU as the activation function for the middle layers. Use Softmax for the output layer because each output has a single lable and all the label probabilities add up to 1.
1. Compile the model using `adam` as the optimizer and `sparse_categorical_crossentropy` as the loss function. For metrics, use `accuracy` for now.
1. Fit the training data.
1. Evaluate your neural network model with the test data.
1. Save your model as `tic-tac-toe.model`.

In [2]:
indices = np.arange(len(y))
train_val_idx, test_idx = train_test_split(indices, test_size=0.2, stratify=y, random_state=SEED)
train_idx, val_idx = train_test_split(train_val_idx, test_size=0.2, stratify=y[train_val_idx], random_state=SEED)
X_train, X_val, X_test = X[train_idx], X[val_idx], X[test_idx]
y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]
print("Train/validation/test:", len(train_idx), len(val_idx), len(test_idx))
print("Majority-class test accuracy:", np.mean(y_test == np.bincount(y_train).argmax()))

def build_model(widths=(32, 16), learning_rate=0.001):
    tf.keras.utils.set_random_seed(SEED)
    model = tf.keras.Sequential([tf.keras.Input(shape=(27,))] +
        [tf.keras.layers.Dense(width, activation="relu") for width in widths] +
        [tf.keras.layers.Dense(2, activation="softmax")])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

model = build_model()
model.summary()
history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                    epochs=30, batch_size=32, verbose=0)
baseline_metrics = model.evaluate(X_test, y_test, verbose=0, return_dict=True)
print("Baseline test:", baseline_metrics)
# Keras 3 requires .keras for reloadable models. Keep the requested directory name.
MODEL_DIR = DATA_DIR / "tic-tac-toe.model"
MODEL_DIR.mkdir(exist_ok=True)
model.save(MODEL_DIR / "baseline.keras")

Train/validation/test: 612 154 192
Majority-class test accuracy: 0.6510416666666666


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 2)              │            34 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,458 (5.70 KB)

 Trainable params: 1,458 (5.70 KB)

 Non-trainable params: 0 (0.00 B)

2026-09-05 16:40:18.149764: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


Baseline test: {'accuracy': 0.9635416865348816, 'loss': 0.15966647863388062}


## Step 3: Make Predictions

Now load your saved model and use it to make predictions on a few random rows in the test dataset. Check if the predictions are correct.

In [3]:
loaded_model = tf.keras.models.load_model(MODEL_DIR / "baseline.keras")
np.testing.assert_allclose(model.predict(X_test, verbose=0),
                           loaded_model.predict(X_test, verbose=0), atol=1e-7)
rng = np.random.default_rng(SEED)
positions = rng.choice(len(test_idx), size=10, replace=False)
probabilities = loaded_model.predict(X_test[positions], verbose=0)
predictions = probabilities.argmax(axis=1)
examples = df.iloc[test_idx[positions]][board_columns].copy()
examples["actual_X_wins"] = y_test[positions].astype(bool)
examples["predicted_X_wins"] = predictions.astype(bool)
examples["P(X wins)"] = probabilities[:, 1]
examples["correct"] = predictions == y_test[positions]
display(examples)
print("Correct sampled predictions:", examples["correct"].sum(), "/", len(examples))

,TL,TM,TR,ML,MM,MR,BL,BM,BR,actual_X_wins,predicted_X_wins,P(X wins),correct
50,x,x,x,b,x,o,o,o,b,True,True,0.998903,True
934,b,b,o,x,x,o,x,b,o,False,False,0.289463,True
38,x,x,x,o,b,o,x,b,o,True,True,0.888488,True
717,x,b,o,x,x,o,o,x,o,False,False,0.039885,True
617,b,b,o,o,b,b,x,x,x,True,True,0.999884,True
800,o,x,b,b,o,x,x,b,o,False,False,0.015582,True
153,x,o,o,x,x,o,b,b,x,True,True,0.964326,True
105,x,x,b,o,x,o,o,x,b,True,True,0.978748,True
191,x,o,b,x,x,o,x,b,o,True,True,0.998378,True
46,x,x,x,o,b,b,o,o,x,True,True,0.990386,True


Correct sampled predictions: 10 / 10


## Step 4: Improve Your Model

Did your model achieve low loss (<0.1) and high accuracy (>0.95)? If not, try to improve your model.

But how? There are so many things you can play with in Tensorflow and in the next challenge you'll learn about these things. But in this challenge, let's just do a few things to see if they will help.

* Add more layers to your model. If the data are complex you need more layers. But don't use more layers than you need. If adding more layers does not improve the model performance you don't need additional layers.
* Adjust the learning rate when you compile the model. This means you will create a custom `tf.keras.optimizers.Adam` instance where you specify the learning rate you want. Then pass the instance to `model.compile` as the optimizer.
    * `tf.keras.optimizers.Adam` [reference](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam).
    * Don't worry if you don't understand what the learning rate does. You'll learn about it in the next challenge.
* Adjust the number of epochs when you fit the training data to the model. Your model performance continues to improve as you train more epochs. But eventually it will reach the ceiling and the performance will stay the same.

In [4]:
# Compare planned changes on validation data; test scores do not select the winner.
configs = [
    ("baseline_30", (32, 16), 0.001, 30),
    ("longer_training", (32, 16), 0.001, 200),
    ("deeper", (32, 16, 16), 0.001, 200),
    ("higher_learning_rate", (32, 16), 0.003, 200),
    ("wider", (64, 32), 0.001, 200),
]
results, candidates = [], {}
for name, widths, lr, epochs in configs:
    candidate = build_model(widths, lr)
    callback = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=25,
                                                restore_best_weights=True)
    h = candidate.fit(X_train, y_train, validation_data=(X_val, y_val),
                      epochs=epochs, batch_size=32, callbacks=[callback], verbose=0)
    metrics = candidate.evaluate(X_val, y_val, verbose=0, return_dict=True)
    results.append(dict(experiment=name, layers=str(widths), learning_rate=lr,
                        epochs_run=len(h.history["loss"]), best_epoch=int(np.argmin(h.history["val_loss"]))+1,
                        val_loss=metrics["loss"], val_accuracy=metrics["accuracy"]))
    candidates[name] = candidate
comparison = pd.DataFrame(results).sort_values("val_loss").reset_index(drop=True)
display(comparison)
best_name = comparison.loc[0, "experiment"]
best_model = candidates[best_name]
final_metrics = best_model.evaluate(X_test, y_test, verbose=0, return_dict=True)
print("Selected on validation loss:", best_name)
print("Final test:", final_metrics)
print("Meets loss < 0.1 and accuracy > 0.95:", final_metrics["loss"] < .1 and final_metrics["accuracy"] > .95)
final_predictions = best_model.predict(X_test, verbose=0).argmax(axis=1)
print(classification_report(y_test, final_predictions, target_names=["X does not win", "X wins"]))
display(pd.DataFrame(confusion_matrix(y_test, final_predictions),
                     index=["actual non-win", "actual win"], columns=["predicted non-win", "predicted win"]))
best_model.save(MODEL_DIR / "best.keras")
reloaded_best = tf.keras.models.load_model(MODEL_DIR / "best.keras")
np.testing.assert_allclose(best_model.predict(X_test, verbose=0),
                           reloaded_best.predict(X_test, verbose=0), atol=1e-7)
comparison.to_csv(DATA_DIR / "model-comparison.csv", index=False)

2026-09-05 16:40:24.453372: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


2026-09-05 16:40:32.061523: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


,experiment,layers,learning_rate,epochs_run,best_epoch,val_loss,val_accuracy
0,wider,"(64, 32)",0.001,126,101,0.061033,0.974026
1,higher_learning_rate,"(32, 16)",0.003,74,49,0.086164,0.980519
2,longer_training,"(32, 16)",0.001,132,107,0.089286,0.974026
3,deeper,"(32, 16, 16)",0.001,69,44,0.124015,0.980519
4,baseline_30,"(32, 16)",0.001,30,30,0.154279,0.954545


Selected on validation loss: wider
Final test: {'accuracy': 0.9791666865348816, 'loss': 0.0959717407822609}
Meets loss < 0.1 and accuracy > 0.95: True
                precision    recall  f1-score   support

X does not win       0.98      0.96      0.97        67
        X wins       0.98      0.99      0.98       125

      accuracy                           0.98       192
     macro avg       0.98      0.97      0.98       192
  weighted avg       0.98      0.98      0.98       192



,predicted non-win,predicted win
actual non-win,64,3
actual win,1,124


**Which approach(es) did you find helpful to improve your model performance?**

Results will be summarized after execution.